In [20]:
import argparse
import numpy as np
import pandas as pd

In [1]:
import pickle
import numpy as np

with open("MythSubspaceSBERT.pkl", "rb") as f:
    obj = pickle.load(f)

print(type(obj))
if isinstance(obj, dict):
    print(obj.keys())
elif isinstance(obj, np.ndarray):
    print(obj.shape)

<class 'numpy.ndarray'>
(768,)


In [21]:
from Config import MYTH_TYPES, MYTH_PAIRS, DEMOGRAPHIC_FEATURES, NLI_CSV_SAMPLE, get_output_dir
from Embeddings import get_subspace, get_myth_unit_vecs
from StatisticalTests import paired_test, independent_test

subspace       = get_subspace()
myth_unit_vecs = get_myth_unit_vecs()

In [22]:
PROJECTION_METRICS = [
    ("proj_subspace_delta",  "proj_subspace_delta"),
    ("proj_myth_unit_delta", "proj_myth_unit_delta"),
]

ORIGINAL_METRICS = [
    ("proj_subspace",  "proj_subspace"),
    ("proj_myth_unit", "proj_myth_unit"),
]

In [ ]:
# ── ContextAppend: paired test against zero baseline ─────────────────────────
if args.append:
    print(f"\nLoading ContextAppend from {args.append}")
    append_df = pd.read_pickle(args.append)
    print(f"  Rows: {len(append_df)}")

    results = []

    # Single myths
    for myth in MYTH_TYPES:
        sub = append_df[append_df["myth_type"] == myth]
        for metric_name, col in PROJECTION_METRICS:
            vals = sub[col].dropna().tolist()
            if len(vals) < 5:
                continue
            r = paired_test(vals, [0.0] * len(vals))
            r.update({
                "myth":    myth,
                "metric":  metric_name,
                "is_pair": False,
                "n":       len(vals),
                "mean":    sub[col].mean(),
                "median":  sub[col].median(),
            })
            results.append(r)

    # Myth pairs (proj_subspace_delta only — no per-myth unit vector for pairs)
    for myth_pair in MYTH_PAIRS:
        sub  = append_df[append_df["myth_pair"] == myth_pair]
        col  = "proj_subspace_delta"
        vals = sub[col].dropna().tolist()
        if len(vals) < 5:
            continue
        r = paired_test(vals, [0.0] * len(vals))
        r.update({
            "myth":    myth_pair,
            "metric":  col,
            "is_pair": True,
            "n":       len(vals),
            "mean":    sub[col].mean(),
            "median":  sub[col].median(),
        })
        results.append(r)

    out = OUT_DIR / "ContextAppend_MythAlignment_StatTests.csv"
    pd.DataFrame(results).to_csv(out, index=False)
    print(f"  Saved: {out.name}")

In [ ]:
# ── Original: independent test vs neutral baseline ────────────────────────────
if args.original:
    print(f"\nLoading Original-NoContextAppend from {args.original}")
    original_df = pd.read_pickle(args.original)
    print(f"  Rows: {len(original_df)}")

    # Build neutral baseline from NLI CSV + T1 embeddings in original_df
    print(f"  Building neutral baseline from {args.nli}")
    nli_df   = pd.read_csv(args.nli)
    myth_nli = nli_df[nli_df["myth_category"] == "MYTH"]
    nli_labels = (
        myth_nli.groupby("narrative_index")
        .apply(lambda g: dict(zip(g["myth_type"], g["overall_label"])))
        .to_dict()
    )

    # One T1 embedding per (narrative_idx, model) — deduplicated
    t1_lookup = (
        original_df[["narrative_idx", "model", "sbert_t1"]]
        .drop_duplicates(["narrative_idx", "model"])
    )

    neutral_rows = []
    for _, row in t1_lookup.iterrows():
        idx    = int(row["narrative_idx"])
        labels = nli_labels.get(idx, {})
        for myth_type in MYTH_TYPES:
            if labels.get(myth_type, "neutral") != "neutral":
                continue
            t1_vec    = np.array(row["sbert_t1"])
            proj_sub  = float(np.dot(t1_vec, subspace))  if subspace is not None else np.nan
            proj_myth = (
                float(np.dot(t1_vec, myth_unit_vecs[myth_type]))
                if myth_type in myth_unit_vecs else np.nan
            )
            neutral_rows.append({
                "narrative_idx":  idx,
                "model":          row["model"],
                "myth_type":      myth_type,
                "proj_subspace":  proj_sub,
                "proj_myth_unit": proj_myth,
            })

    neutral_df = pd.DataFrame(neutral_rows)
    print(f"  Neutral baseline rows: {len(neutral_df)}")

    results = []
    for myth in MYTH_TYPES:
        grp_neu = neutral_df[neutral_df["myth_type"] == myth]
        for label in ["entailment", "contradiction"]:
            grp_org = original_df[
                (original_df["myth_type"] == myth) &
                (original_df["narrative_nli_label"] == label)
            ]
            if len(grp_org) < 5 or len(grp_neu) < 5:
                continue
            for metric_name, col in ORIGINAL_METRICS:
                r = independent_test(
                    grp_org[col].dropna().tolist(),
                    grp_neu[col].dropna().tolist(),
                )
                r.update({
                    "myth":          myth,
                    "organic_label": label,
                    "metric":        metric_name,
                    "mean_original": grp_org[col].mean(),
                    "mean_neutral":  grp_neu[col].mean(),
                })
                results.append(r)

    out = OUT_DIR / "ExpOriginal-NoContextAppend_MythAlignment_IndependentGroupComparison.csv"
    pd.DataFrame(results).to_csv(out, index=False)
    print(f"  Saved: {out.name}")